# 学習済みのニューラルモデルでかな漢字変換をする (zenz)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/31_japanese_input/kkc_neural.ipynb)

NLP クックブック 31-10 ／ 日本語入力（かな漢字変換）

大量のテキストで学習されたニューラルかな漢字変換モデルを使って、文脈を踏まえた高精度な変換をしたい。

In [1]:
!pip install -q jaconv

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Miwa-Keita/zenz-v2.5-small"   # かな漢字変換用に学習された GPT-2 型のモデル (約 9,000 万パラメータ)。CPU でも動く
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def convert(kana, n=1, num_beams=None):
    """入力は「 + カタカナの読み + 」の形で与え、続きとして変換結果を生成させる"""
    prompt = "" + kana + ""
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    out = model.generate(**enc, max_new_tokens=len(kana) + 10, do_sample=False, num_beams=num_beams or max(n, 1),
                         num_return_sequences=n, pad_token_id=tokenizer.eos_token_id)
    results = [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]
    return results[0] if n == 1 else results

for kana in ["キョウハイイテンキデス", "シゼンゲンゴショリノケンキュウヲスル", "ハシノウエデハシヲオトシタ", "キシャノキシャガキシャデキシャシタ",
             "カレハイシャニイシャリョウヲセイキュウシタ", "スマホデドウガヲミル", "コノセンセイハジンコウチノウノセンモンカダ"]:
    print(f"{kana} → {convert(kana)}")

キョウハイイテンキデス → 今日はいい天気です
シゼンゲンゴショリノケンキュウヲスル → 自然言語処理の研究をする
ハシノウエデハシヲオトシタ → 橋の上で橋を落とした
キシャノキシャガキシャデキシャシタ → 記者の記者が記者で記者した
カレハイシャニイシャリョウヲセイキュウシタ → 彼は医者に慰謝料を請求した
スマホデドウガヲミル → スマホで動画を見る
コノセンセイハジンコウチノウノセンモンカダ → この先生は人工知能の専門家だ


In [3]:
# 候補を複数出す (ビームサーチ)
for kana in ["コウエンニイク", "カミヲキル", "アツイヒガツヅク"]:
    print(kana, convert(kana, n=4))

コウエンニイク ['公園に行く', '公園にいく', '公演に行く', '講演に行く']
カミヲキル ['髪を切る', '紙を切る', '上を切る', '髪をきる']
アツイヒガツヅク ['暑い日が続く', '熱い日が続く', '暑い日がつづく', 'あつい日が続く']


In [4]:
# ひらがなの入力はカタカナに直して渡す / 速度を測る
import time
import jaconv

text = "わたしはまいあさこーひーをのんでからだいがくにいきます"
t = time.perf_counter()
print(convert(jaconv.hira2kata(text)), f"({time.perf_counter() - t:.2f} 秒)")

私は毎朝コーヒーを飲んでから大学に行きます (0.90 秒)


- zenz は、日本語入力アプリ azooKey のために作られたモデル。読みを条件として、変換後の文を 1 文字ずつ生成する。大量のテキストから学習しているので、現代の語彙や長い文脈に強い。
- 単語 bigram のモデル（この章の「前後の単語を考慮してかな漢字変換をする（bigram 言語モデル）」）と違い、文全体を見て同音異義語を選べる（「医者に慰謝料を請求した」）。ただし万能ではなく、「はしのうえではしをおとした」は「橋の上で橋を落とした」になり、2 つ目の「はし」に「箸」を選べない。「きしゃのきしゃがきしゃできしゃした」のような同音異義語の言葉遊びも、すべて同じ表記になってしまう。
- 生成モデルなので、読みと対応しない文字を出力することがある。実用のシステムでは、従来型の変換結果を候補として作り、ニューラルモデルで検証・並べ替えをする構成（azooKey の Zenzai）がとられている。
- 入力のたびにモデルを動かすので、応答の速さが課題になる。小さなモデル、量子化、入力途中の計算結果の再利用などで対処する。